In [119]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
import warnings
warnings.filterwarnings('ignore')

###Reading_Data

In [120]:
df=pd.read_csv(r"/content/WA_Fn-UseC_-Telco-Customer-Churn.csv",encoding="latin_1")

In [121]:
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [122]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


###Data  Clean & Clansing

In [123]:
df.drop("customerID",axis=1,inplace=True)

In [124]:
df["TotalCharges"] = df["TotalCharges"].replace(" ", np.nan)
df['TotalCharges']=pd.to_numeric(df['TotalCharges'])

In [125]:
df["TotalCharges"].isnull().sum()

np.int64(11)

In [126]:
df["MultipleLines"].replace("No phone service","No")

,MultipleLines
0,No
1,No
2,No
3,No
4,No
5,Yes
6,Yes
7,No
8,Yes
9,No


In [127]:
tran=["Partner","Dependents","PhoneService","MultipleLines","OnlineSecurity","OnlineBackup","TechSupport","StreamingTV","StreamingMovies","PaperlessBilling","Churn","DeviceProtection"]
for col in tran:
  df[col]=df[col].map({"Yes":1,"No":0})


In [128]:
df.dtypes

,0
gender,object
SeniorCitizen,int64
Partner,int64
Dependents,int64
tenure,int64
PhoneService,int64
MultipleLines,float64
InternetService,object
OnlineSecurity,float64
OnlineBackup,float64


###split input & output

In [129]:
input = df.drop("Churn",axis=1)
ouput = df["Churn"]

###Train and Test For Data

In [130]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(input,ouput,test_size=0.2,random_state=42)

###Pipeline & OneHotEncoding & Simpleimuter

In [131]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
si=SimpleImputer(strategy="median")
pr_data=Pipeline(steps=[("imputer",si)])
X_train[["TotalCharges"]]=si.fit_transform(X_train[["TotalCharges"]])
pr_data

Pipeline(steps=[('imputer', SimpleImputer(strategy='median'))])

In [149]:
from sklearn.preprocessing import StandardScaler
ss=StandardScaler()
col =["tenure","MonthlyCharges","TotalCharges"]
X_train[col]=ss.fit_transform(X_train[col])
X_test[col]=ss.transform(X_test[col])
num_data=Pipeline(steps=[("pr_data",SimpleImputer()),("scaler",ss)])
num_data


Pipeline(steps=[('pr_data', SimpleImputer()), ('scaler', StandardScaler())])

In [150]:
from sklearn.preprocessing import OneHotEncoder
ohe = OneHotEncoder(drop="first", handle_unknown="ignore")
cat_data = Pipeline(steps=[("onehotEncoder", ohe)])
cols = ["gender", "Contract", "InternetService", "PaymentMethod"]
X_train_encoded = categ_data.fit_transform(X_train[cols])
X_test_encoded = categ_data.transform(X_test[cols])

cat_data

Pipeline(steps=[('onehotEncoder',
                 OneHotEncoder(drop='first', handle_unknown='ignore'))])

In [154]:
from sklearn.compose import ColumnTransformer
pre_pepline=ColumnTransformer(transformers=[("num",num_data,col),("cat",cat_data,cols)])
pre_pepline

ColumnTransformer(transformers=[('num',
                                 Pipeline(steps=[('pr_data', SimpleImputer()),
                                                 ('scaler', StandardScaler())]),
                                 ['tenure', 'MonthlyCharges', 'TotalCharges']),
                                ('cat',
                                 Pipeline(steps=[('onehotEncoder',
                                                  OneHotEncoder(drop='first',
                                                                handle_unknown='ignore'))]),
                                 ['gender', 'Contract', 'InternetService',
                                  'PaymentMethod'])])

###ML

In [135]:
from sklearn.linear_model import LogisticRegression
lr=LogisticRegression()
lr.fit(X_train_encoded,y_train)


LogisticRegression()

In [136]:
y_test_pred=lr.predict(X_test_encoded)
y_test_pred

array([0, 0, 0, ..., 0, 0, 0])

In [137]:
from sklearn.metrics import accuracy_score,confusion_matrix,classification_report
print(accuracy_score(y_test,y_test_pred))
print(confusion_matrix(y_test,y_test_pred))
print(classification_report(y_test,y_test_pred))

0.7906316536550745
[[928 108]
 [187 186]]
              precision    recall  f1-score   support

           0       0.83      0.90      0.86      1036
           1       0.63      0.50      0.56       373

    accuracy                           0.79      1409
   macro avg       0.73      0.70      0.71      1409
weighted avg       0.78      0.79      0.78      1409



In [138]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GridSearchCV
dt=DecisionTreeClassifier()
para_grid={"max_depth":np.arange(3,10),"min_samples_split":range(2,15),"criterion":["gini","entropy"]}
dt_cv=(GridSearchCV(dt,para_grid,cv=5))
dt_cv.fit(X_train_encoded,y_train)
dt_pred=dt_cv.predict(X_test_encoded)


In [139]:
print(dt_cv.best_score_,dt_cv.best_params_)

0.7701458311334417 {'criterion': 'gini', 'max_depth': np.int64(4), 'min_samples_split': 2}


In [140]:
from sklearn.metrics import accuracy_score,confusion_matrix,classification_report
print(accuracy_score(y_test,dt_pred))
print(confusion_matrix(y_test,dt_pred))
print(classification_report(y_test,dt_pred))

0.7849538679914834
[[933 103]
 [200 173]]
              precision    recall  f1-score   support

           0       0.82      0.90      0.86      1036
           1       0.63      0.46      0.53       373

    accuracy                           0.78      1409
   macro avg       0.73      0.68      0.70      1409
weighted avg       0.77      0.78      0.77      1409



In [141]:
from sklearn.ensemble import RandomForestClassifier
rf=RandomForestClassifier()
para_grid={"max_depth":np.arange(3,10),"n_estimators":range(10,25)}
rf_cv=(GridSearchCV(rf,para_grid,cv=5))
rf_cv.fit(X_train_encoded,y_train)
rf_pred=rf_cv.predict(X_test_encoded)

In [142]:
print(rf_cv.best_score_,rf_cv.best_params_)

0.7712115504940101 {'max_depth': np.int64(7), 'n_estimators': 17}


In [143]:
from sklearn.metrics import accuracy_score,confusion_matrix,classification_report
print(accuracy_score(y_test,rf_pred))
print(confusion_matrix(y_test,rf_pred))
print(classification_report(y_test,rf_pred))

0.7906316536550745
[[928 108]
 [187 186]]
              precision    recall  f1-score   support

           0       0.83      0.90      0.86      1036
           1       0.63      0.50      0.56       373

    accuracy                           0.79      1409
   macro avg       0.73      0.70      0.71      1409
weighted avg       0.78      0.79      0.78      1409



In [144]:
from sklearn.neighbors import KNeighborsClassifier
knn=KNeighborsClassifier()
param_grid={'n_neighbors':np.arange(1,10)}
knn_cv=GridSearchCV(knn,param_grid,cv=5)
knn_cv.fit(X_train_encoded,y_train)
yk_pred=knn_cv.predict(X_test_encoded)


In [145]:
from sklearn.metrics import accuracy_score,confusion_matrix,classification_report
print(accuracy_score(y_test,yk_pred))
print(confusion_matrix(y_test,yk_pred))
print(classification_report(y_test,yk_pred))

0.772888573456352
[[986  50]
 [270 103]]
              precision    recall  f1-score   support

           0       0.79      0.95      0.86      1036
           1       0.67      0.28      0.39       373

    accuracy                           0.77      1409
   macro avg       0.73      0.61      0.63      1409
weighted avg       0.76      0.77      0.74      1409



In [155]:
final_pipeline=Pipeline(steps=[("pre_pepline",pre_pepline),("lr",LogisticRegression())])
final_pipeline


Pipeline(steps=[('pre_pepline',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('pr_data',
                                                                   SimpleImputer()),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['tenure', 'MonthlyCharges',
                                                   'TotalCharges']),
                                                 ('cat',
                                                  Pipeline(steps=[('onehotEncoder',
                                                                   OneHotEncoder(drop='first',
                                                                                 handle_unknown='ignore'))]),
                                                  ['gender', 'Contract',
                                                   'InternetService',
                                                   'PaymentMethod'])])),
                ('lr', LogisticRegression())])

In [156]:
final_pipeline.fit(X_train,y_train)

Pipeline(steps=[('pre_pepline',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('pr_data',
                                                                   SimpleImputer()),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['tenure', 'MonthlyCharges',
                                                   'TotalCharges']),
                                                 ('cat',
                                                  Pipeline(steps=[('onehotEncoder',
                                                                   OneHotEncoder(drop='first',
                                                                                 handle_unknown='ignore'))]),
                                                  ['gender', 'Contract',
                                                   'InternetService',
                                                   'PaymentMethod'])])),
                ('lr', LogisticRegression())])

In [158]:
y_pred = final_pipeline.predict(X_test)
y_pred

array([1, 0, 0, ..., 0, 0, 0])

In [160]:
print('evaluation on training \n', classification_report(y_train, final_pipeline.predict(X_train)))
print('evaluation on testing \n', classification_report(y_test, y_pred))

evaluation on training 
               precision    recall  f1-score   support

           0       0.84      0.89      0.86      4138
           1       0.64      0.52      0.57      1496

    accuracy                           0.79      5634
   macro avg       0.74      0.71      0.72      5634
weighted avg       0.78      0.79      0.79      5634

evaluation on testing 
               precision    recall  f1-score   support

           0       0.85      0.90      0.88      1036
           1       0.67      0.56      0.61       373

    accuracy                           0.81      1409
   macro avg       0.76      0.73      0.74      1409
weighted avg       0.80      0.81      0.81      1409



In [161]:
from sklearn.metrics import accuracy_score

acc = accuracy_score(y_test, y_pred)
print("Accuracy:", acc)

Accuracy: 0.8119233498935415


In [162]:
import joblib
joblib.dump(final_pipeline,'final_model.pkl')

['final_model.pkl']

In [163]:
model = joblib.load("final_model.pkl")

In [164]:
model.predict(X_test)

array([1, 0, 0, ..., 0, 0, 0])